# 01. Paraphrase pool and unit manifests

**Design decisions.**
- The paraphrase pool (`suite_data/paraphrases.json`) holds model-written variants of the 18 boilerplate sentences and the 29 query templates (`src/paraphrase.py`), generated once by Qwen2.5-7B-Instruct through vLLM at temperature 0.8 and kept only when every placeholder survives, the wording differs from the original and the length stays within half to twice the original. The model never sees a record or a secret: it rewrites templates. If the file exists the generation is skipped (record once).
- Unit manifests (`suite_data/manifest_<split>.json`) list every unit of every split (probe 600, legit 300, auth 200; train 2000, calibration 3000, utility 1000, test 2000) without its worlds: the uid, the seed, the pool hash, the requester, the target, the family, the query. A world is rebuilt from the manifest row by `world.build_unit(split, index, pool)`; notebook 02 asserts that the rebuilt unit matches its row. Every leak unit passes the low-equivalence assertion (Equation 2) here.
- Domain 2 (Enron with SARA labels) is built in notebook 07, where it is used (amendment to plan v1).
- Runtime: L4 for the paraphrase stage (a few minutes), CPU for the rest. With `REFUSELEAK_DRY_RUN=1` the pool holds deterministic rewrites and no server starts.


In [ ]:
# === SETTINGS ===
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
VLLM_PORT       = 8000
VLLM_GPU_FRAC   = 0.90
MAX_MODEL_LEN   = 4096
VLLM_EXTRA_ARGS = ["--generation-config", "vllm"]
N_PARAPHRASES   = 6             # variants kept per template
PARAPHRASE_MODEL = "qwen25_7b"

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, json, shutil, subprocess, importlib, time
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/REFUSELEAK_Research")
REPO_DIR   = DRIVE_ROOT / "refuseleak-research"
CRED       = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["REFUSELEAK_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["REFUSELEAK_REPO_ROOT"] = str(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

## Stage 1. Install vLLM (L4 runtime for this stage and Stage 2)

In [ ]:
# === STAGE: install ===
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print("pip", *a, "-> rc", r.returncode); print(r.stderr[-800:])
    return r.returncode
_ = pip('vllm', 'openai')

## Stage 2. Paraphrase pool (record once)

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 2: the paraphrase pool, generated once ===
import json
import config, world as W, paraphrase as PP, serving
for m in (config, W, PP, serving): importlib.reload(m)
config.ensure_dirs()
pool_path = config.SUITE_DATA / "paraphrases.json"
if pool_path.exists():
    print("pool present:", pool_path)
else:
    server = serving.Server(port=VLLM_PORT, gpu_frac=VLLM_GPU_FRAC, max_model_len=MAX_MODEL_LEN, extra_args=VLLM_EXTRA_ARGS)
    server.ensure(PARAPHRASE_MODEL)
    llm = serving.make_llm(PARAPHRASE_MODEL, port=VLLM_PORT)
    raw = PP.generate(llm, n_per=N_PARAPHRASES, seed=config.SEED)
    PP.write(raw, pool_path)
    server.stop()
pool = W.load_pool(pool_path)
summary = {"pool_hash": W.pool_hash(pool), "model": config.MODELS[PARAPHRASE_MODEL], "dry_run": serving.DRY_RUN, **PP.summary(pool)}
print(json.dumps(summary, indent=2))
show = [{"section": sec, "key": key, "original": orig, "variants": " | ".join(v[1:4])}
        for sec in ("queries", "boilerplate") for key, d in pool[sec].items() for orig, v in list(d.items())[:1]]
display(pd.DataFrame(show))
(config.REPORTS / "01_paraphrases.json").write_text(json.dumps(summary, indent=2))

## Stage 3. Unit manifests for every split

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 3: unit manifests for every split, with the low-equivalence assertion on every leak unit ===
import json, time
import config, world as W, acl
for m in (config, W, acl): importlib.reload(m)
pool = W.load_pool()
phash = W.pool_hash(pool)
stats = {"pool_hash": phash, "world_version": config.WORLD_VERSION, "splits": {}}
t0 = time.time()
for split, n in config.SPLITS.items():
    rows = []
    for i in range(n):
        u = W.build_unit(split, i, pool)
        if u["kind"] == "leak":
            assert acl.low_equivalent(u["worlds"]["0"], u["worlds"]["1"], u["requester"]), u["uid"]
        rows.append(W.manifest_row(u))
    (config.SUITE_DATA / f"manifest_{split}.json").write_text(json.dumps(rows, indent=0))
    df = pd.DataFrame(rows)
    stats["splits"][split] = {"units": n, "kind": df["kind"].iloc[0],
                              "families": df["family"].value_counts().to_dict(),
                              "secrets": df["secret"].value_counts().to_dict() if "secret" in df else {},
                              "query_words_mean": round(float(df["query"].str.split().str.len().mean()), 2),
                              "distinct_queries": int(df["query"].nunique())}
    print(f"{split}: {n} units, {time.time() - t0:.0f} s")
stats["seconds"] = round(time.time() - t0, 1)
(config.REPORTS / "01_world_stats.json").write_text(json.dumps(stats, indent=2))
display(pd.DataFrame({s: v["families"] for s, v in stats["splits"].items()}).fillna(0).astype(int))

## Stage 4. Samples by family

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 4: what the requests look like, by family ===
import json, config
probe = pd.DataFrame(json.loads((config.SUITE_DATA / "manifest_probe.json").read_text()))
legit = pd.DataFrame(json.loads((config.SUITE_DATA / "manifest_legit.json").read_text()))
auth = pd.DataFrame(json.loads((config.SUITE_DATA / "manifest_auth.json").read_text()))
cols = ["uid", "secret", "family", "query", "target_ref", "codename"]
display(probe.groupby("family").head(2)[cols].sort_values("family"))
display(legit.groupby("family").head(3)[["uid", "family", "query", "target_ref"]])
display(auth.head(3)[["uid", "family", "query", "target_ref"]])

## End of unit: commit

Save the notebook first (File > Save), then run the cell below.

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
    'nb01: paraphrase pool (suite_data/paraphrases.json, Qwen2.5-7B rewrites of the templates) and unit manifests for every split (suite_data/manifest_*.json; every leak unit low-equivalent); reports/01_paraphrases.json, 01_world_stats.json'],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr)